# Part 18 (실습) — 최종 프로젝트: 문서 제작 파이프라인 🎓

> **이 노트북의 목표**
> 18개 파트의 모든 것을 하나로 묶음. **그래프(뼈대) + 조사 에이전트(RAG) + 작성·검토 + 루프 + HITL 승인 + 안전장치**를 결합한 운영 가능한 문서 제작 파이프라인을 처음부터 끝까지 구현함.
>
> **사용 모델**: 채팅 `gemini-3-flash`, 임베딩 `gemini-embedding-001` · **선행**: Part 0~17 (전부)

## 0. 준비

In [ ]:
# ─── 패키지 설치 ───
# %pip install: 주피터 노트북 안에서 파이썬 패키지를 설치하는 매직 명령어
#   -q (quiet): 설치 로그 최소화 | -U (upgrade): 최신 버전으로 업그레이드
%pip install -q -U langchain langchain-google-genai langchain-text-splitters langgraph

In [ ]:
# os: 파이썬 내장 모듈 — 환경 변수(GOOGLE_API_KEY 등)를 읽고 쓸 때 사용
import os
# getpass: 입력한 글자를 화면에 숨겨서 받는 함수 (비밀번호 입력처럼)
#   → API 키를 노출 없이 안전하게 입력받기 위해 사용
from getpass import getpass
if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass("Gemini API 키: ")
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
# ─── Gemini 모델 인스턴스 생성 ───
# model: 사용할 Gemini 모델명 (예: "gemini-3-flash" — 빠르고 저렴한 기본 모델)
# temperature: 답변의 무작위성 조절 (0=일관된 답 / 1=창의적·다양한 답)
model = ChatGoogleGenerativeAI(model="gemini-3-flash", temperature=0.3, max_retries=3, timeout=30)  # 신뢰성(17)
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")
print("준비 완료")

## 1. 상태 정의 (Part 13)

파이프라인이 들고 다닐 상태. `history`는 누적(리듀서).

In [ ]:
# typing: 파이썬 타입 힌트 모듈 — 함수/변수의 타입을 명시하여 코드 가독성 향상
#   → TypedDict: 딕셔너리의 키와 값 타입을 정의하는 클래스
#   → Annotated: 타입에 추가 메타데이터(리듀서 등)를 붙이는 데 사용
from typing import TypedDict, Annotated
# operator: 파이썬 내장 모듈 — 연산자를 함수로 쓸 수 있게 해줌
#   → operator.add: 리스트끼리 합치는 리듀서 함수로 자주 사용됨
import operator

class DocState(TypedDict):
    topic: str
    research: str        # 조사 결과
    draft: str           # 현재 초안
    review: str          # 검토 결과
    approved: bool       # 사람 승인 여부
    attempts: int        # 작성 시도 횟수
    history: Annotated[list, operator.add]

print("상태 정의 완료")

## 2. RAG 준비 (Part 8) — 조사 에이전트가 쓸 지식

In [ ]:
# RecursiveCharacterTextSplitter: 긴 문서를 작은 청크(조각)로 나누는 도구
#   → chunk_size: 각 청크의 최대 글자 수
#   → chunk_overlap: 청크 사이에 겹치는 글자 수 (맥락 유지용)
#   → 재귀적으로 문단→문장→단어 순서로 자연스럽게 분할
from langchain_text_splitters import RecursiveCharacterTextSplitter
# InMemoryVectorStore: 메모리에 벡터(임베딩)를 저장하는 간단한 벡터스토어
#   → 문서를 벡터로 변환해 저장하고, 유사도 기반 검색을 수행
#   → 프로토타입/학습용 (프로덕션에서는 Chroma, Pinecone 등 사용)
from langchain_core.vectorstores import InMemoryVectorStore
# ChatPromptTemplate: 채팅 모델용 메시지 템플릿을 만드는 클래스
#   → .from_messages([(역할, 템플릿), ...]): 역할별 메시지 리스트 생성
#   → 역할: "system"(시스템), "human"(사용자), "ai"(모델 응답)
from langchain_core.prompts import ChatPromptTemplate
# RunnablePassthrough: 입력을 그대로 통과시키는 부품 (아무것도 안 함)
#   → 파이프라인에서 원본 입력을 다음 단계로 그대로 전달할 때 사용
from langchain_core.runnables import RunnablePassthrough
# StrOutputParser: AIMessage 객체에서 .content(텍스트)만 꺼내주는 파서
#   → 체인의 마지막에 붙여서 순수 문자열 결과를 얻을 때 사용
#   → 사용법: prompt | llm | StrOutputParser()
from langchain_core.output_parsers import StrOutputParser

reference = """생성형 AI 업무 도입 자료.
- 2025년 기업 도입률 전년 대비 2배 증가.
- 반복 업무(문서·요약) 시간 평균 30% 단축.
- 주요 리스크: 환각, 데이터 보안, 저작권.
- 성공 핵심: 명확한 가이드라인과 직원 교육.
- 도입 단계: 파일럿 → 부서 확대 → 전사 적용."""

chunks = RecursiveCharacterTextSplitter(chunk_size=80, chunk_overlap=15).split_text(reference)
store = InMemoryVectorStore.from_texts(chunks, embedding=embeddings)
retriever = store.as_retriever(search_kwargs={"k": 3})
rag = ({"context": retriever | (lambda ds: "\n".join(d.page_content for d in ds)),
        "question": RunnablePassthrough()}
       | ChatPromptTemplate.from_template("자료 근거로 답해.\n[자료]\n{context}\n[질문]\n{question}")
       | model | StrOutputParser())
print("RAG 준비 완료")

## 3. 노드 채우기 — 에이전트·RAG·체인을 자리에 (Part 7·8·9·13)

조사=에이전트+RAG도구, 작성=체인, 검토=모델 판단. 각 노드는 "상태→부분 갱신".

In [ ]:
from langchain.agents import create_agent
# @tool 데코레이터: 일반 파이썬 함수를 LangChain "도구"로 변환
#   → 함수의 docstring이 도구 설명이 되고, 파라미터가 도구 입력 스키마가 됨
#   → 에이전트가 이 설명을 읽고 언제 이 도구를 쓸지 스스로 판단함
from langchain_core.tools import tool

# 조사 노드 = 에이전트 + RAG 도구 (그래프 노드 안의 에이전트 — Part 12 계층)
@tool
def search_reference(query: str) -> str:
    """참고 자료에서 사실·수치를 검색한다. 근거가 필요할 때 사용한다."""
# .invoke(): 입력을 모델/체인에 보내고 결과를 받는 LangChain의 핵심 실행 메서드
#   → 모든 LangChain 부품(Runnable)이 이 메서드를 공유함
    return rag.invoke(query)

research_agent = create_agent(model=model, tools=[search_reference],
    system_prompt="너는 리서치 전문가야. 주제 관련 사실과 수치를 조사해 3줄로 정리해.")

def research_node(state: DocState) -> dict:
    try:
        r = research_agent.invoke({"messages": [("user", f"{state[\'topic\']} 도입 효과와 리스크 조사")]})
        return {"research": r["messages"][-1].content, "history": ["🔬 조사 완료"]}
    except Exception as e:
        return {"research": "(조사 실패)", "history": [f"조사 오류: {e}"]}   # 예외 처리(17)

# 작성 노드 = 체인
def write_node(state: DocState) -> dict:
    fb = f"\n이전 검토의견: {state[\'review\']}" if state.get("review") else ""
    draft = model.invoke(
        f"주제: {state[\'topic\']}\n자료: {state[\'research\']}{fb}\n위 자료의 수치를 인용해 3~4문장 보고서를 써줘."
    ).content
    return {"draft": draft, "attempts": state["attempts"] + 1, "history": [f"✍️ 작성(시도 {state[\'attempts\']+1})"]}

# 검토 노드 = 판단 (근거 수치가 있으면 통과 — 데모)
def review_node(state: DocState) -> dict:
    ok = any(ch.isdigit() for ch in state["draft"])
    return {"review": "통과" if ok else "보강 필요(근거 부족)", "history": [f"🔎 검토 → {'통과' if ok else '반려'}"]}

print("노드 채우기 완료: research(에이전트+RAG), write(체인), review(판단)")

## 4. HITL 승인 + 발행 노드 (Part 14)

In [ ]:
# ─── Human-in-the-Loop (사람 개입) ───
# interrupt: 그래프 실행을 일시 정지하고 사람의 승인/수정을 기다리는 함수
# Command: 일시 정지된 그래프에 사람의 판단(승인/반려/수정)을 전달하는 명령 객체
from langgraph.types import interrupt, Command

def approval_node(state: DocState) -> dict:
    """발행 전 사람 승인 — 비가역 직전에 멈춤"""
    decision = interrupt({"question": "이 보고서를 발행할까요?", "draft": state["draft"]})
    return {"approved": (decision == "승인"), "history": [f"🙋 사람 결정: {decision}"]}

def publish_node(state: DocState) -> dict:
    return {"history": ["📤 발행 완료 ✅"]}

print("HITL 승인 노드 + 발행 노드 정의 완료")

## 5. 그래프 조립 — 뼈대 + 분기 + 루프 + 안전장치 (Part 13·14·17)

In [ ]:
# ─── LangGraph 핵심 클래스 ───
# StateGraph: 노드(작업)와 엣지(연결)로 이루어진 상태 그래프를 직접 설계하는 클래스
#   → 노드: 각 단계에서 실행할 함수
#   → 엣지: 노드 간의 실행 순서 (조건부 분기 가능)
# START, END: 그래프의 시작점과 끝점을 나타내는 특수 상수
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver   # 운영은 SqliteSaver(17)

def route_review(state: DocState) -> str:
    if state["review"] == "통과":
        return "approve"
    if state["attempts"] >= 3:        # 안전장치: 무한 루프 방지(17)
        return "give_up"
    return "revise"

builder = StateGraph(DocState)
# .add_node(): 그래프에 새로운 노드(작업 단계)를 추가
#   → (노드이름, 실행할함수) 형태로 등록
builder.add_node("research", research_node)
builder.add_node("write", write_node)
builder.add_node("review", review_node)
builder.add_node("approval", approval_node)
builder.add_node("publish", publish_node)
# .add_node(): 그래프에 새로운 노드(작업 단계)를 추가
#   → (노드이름, 실행할함수) 형태로 등록
builder.add_node("give_up", lambda s: {"history": ["⛔ 3회 초과 — 검토 중단"]})

# .add_edge(): 두 노드를 순차적으로 연결 (A → B)
builder.add_edge(START, "research")
builder.add_edge("research", "write")
builder.add_edge("write", "review")
# .add_conditional_edges(): 조건에 따라 다른 노드로 분기하는 엣지 추가
#   → 라우팅 함수의 반환값에 따라 다음 노드가 결정됨
builder.add_conditional_edges("review", route_review,
    {"approve": "approval", "revise": "write", "give_up": "give_up"})   # 반려→작성 루프
builder.add_conditional_edges("approval",
    lambda s: "publish" if s["approved"] else "give_up",
    {"publish": "publish", "give_up": "give_up"})
# .add_edge(): 두 노드를 순차적으로 연결 (A → B)
builder.add_edge("publish", END)
builder.add_edge("give_up", END)

# .compile(): 그래프를 실행 가능한 형태로 컴파일 (체크포인터 연결 등)
graph = builder.compile(checkpointer=InMemorySaver())   # 체크포인터(HITL·영속의 토대)
print("그래프 조립·컴파일 완료")
print(graph.get_graph().draw_ascii())

## 6. 실행 — 조사→작성→검토→(루프)→승인 대기

`recursion_limit`(안전장치)와 thread_id로 실행. approval에서 멈춤.

In [ ]:
config = {"configurable": {"thread_id": "doc-final"}, "recursion_limit": 25}   # 안전장치(17)

result = graph.invoke({
    "topic": "생성형 AI 도입", "research": "", "draft": "",
    "review": "", "approved": False, "attempts": 0, "history": []
}, config=config)

print("=== 여기까지의 공정 ===")
for h in result["history"]:
    print(" •", h)
print("\n=== 작성된 초안 ===")
print(result["draft"])
print("\n=== 멈춤: 사람 승인 대기 ===")
print(result["__interrupt__"][0].value["question"])

## 7. 사람 승인 → 발행 (Part 14)

같은 thread_id로 `Command(resume="승인")` → 멈춘 지점부터 발행까지.

In [ ]:
final = graph.invoke(Command(resume="승인"), config=config)

print("=== 전체 공정 완료 ===")
for h in final["history"]:
    print(" •", h)
print("\n→ 조사→작성→검토→승인→발행이 그래프로 보장됨. 18개 파트가 하나로 작동! 🎓")

## 8. 통합 점검 — 어디에 무엇이 쓰였나

In [ ]:
integration = {
    "체인·파서 (3·4)": "write_node의 모델 호출, RAG 체인",
    "도구·RAG (7·8)": "search_reference 도구 + InMemoryVectorStore RAG",
    "에이전트 (9)":    "research_agent (그래프 노드 안의 에이전트)",
    "메모리·영속 (10·17)": "checkpointer (HITL 멈춤·재개의 토대)",
    "StateGraph (13)": "전체 뼈대 — 노드·엣지·조건분기·루프",
    "HITL (14)":       "approval_node의 interrupt/resume",
    "안전장치 (17)":   "recursion_limit + attempts 제한 + 예외 처리",
    "관측 (16)":       "LANGSMITH_TRACING 켜면 전 공정 트리 기록",
}
for k, v in integration.items():
    print(f"✅ {k}: {v}")
print("\n🎓 LangChain 에이전트 마스터 커리큘럼 졸업!")

## ✏️ 졸업 과제

**과제**: 이 파이프라인을 당신의 도메인으로 바꿔보기.
- `reference`를 당신 분야의 자료로 교체
- `topic`을 당신의 실제 주제로
- 필요하면 노드를 추가(예: 번역 노드, 또는 조사를 멀티에이전트로 — Part 15)
- 운영하려면 `InMemorySaver` → `SqliteSaver`로 교체(Part 17)

## 정리 — 졸업 🎓

| 단계 | 결합한 것 |
|---|---|
| 1 | 상태 (13) |
| 2 | RAG (8) |
| 3 | 노드: 에이전트(9)+RAG(7·8)+체인(3·4) |
| 4~5 | 그래프(13)+HITL(14)+안전장치(17) |
| 6~7 | 실행: 루프·승인·발행 |
| 8 | 통합 점검 |

### 3줄 요약
1. 그래프(뼈대) 안에 에이전트·RAG·체인을 계층적으로 결합함.
2. 검토 루프·HITL 승인·안전장치로 흐름이 보장되고 안전함.
3. 18개 파트가 하나의 작동하는 시스템으로 맞춰짐 — 졸업!

### 🎓 커리큘럼 완주
초급(체인)부터 고급(그래프·멀티·운영)까지. 관통 원칙: **올바른 도구를, 올바른 자리에, 필요한 만큼만.** 이제 당신의 문제에 펼칠 차례임.